# Notebook 2: Word2Vec and GloVe — Static Embeddings

> **Article section:** *Part 2 — The Vector Space Revolution*

N-grams treat every word as an isolated symbol. Word2Vec treats words as **points in geometric space** —  
and suddenly, relationships between words become distances between vectors.

---
**What we cover:**
1. Train a Word2Vec model on our corpus using `gensim`
2. Explore nearest neighbours
3. Reproduce the famous King − Man + Woman ≈ Queen arithmetic
4. Plot a PCA projection of the embedding space
5. Demonstrate the key limitation: static embeddings can't handle polysemy
---

In [ ]:
import sys, os, re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.embedding_utils import (
    train_word2vec,
    analogy,
    nearest_neighbors,
    plot_embeddings,
    cosine_similarity,
)

print('Imports OK.')

In [ ]:
# Load and tokenise corpus
def tokenize(text):
    return re.findall(r"[a-z']+", text.lower().strip())

corpus_path = os.path.join('..', 'data', 'sample_corpus.txt')
with open(corpus_path) as f:
    sentences = [tokenize(line) for line in f if line.strip()]

print(f'{len(sentences)} sentences loaded.')

## 1. Train Word2Vec

We're training on a tiny corpus (20 sentences) just for demonstration.  
In real applications, Word2Vec is trained on billions of tokens.

In [ ]:
model = train_word2vec(
    sentences,
    vector_size=50,   # 50-dimensional vectors (production: 300)
    window=5,         # context window of ±5 words
    min_count=1,      # include all words (corpus is tiny)
    epochs=300,       # more epochs since corpus is small
)

# Access the word vectors
wv = model.wv
print(f"\nVector for 'bank' (first 10 dims): {wv['bank'][:10].round(3)}")
print(f"Vector shape: {wv['bank'].shape}")

## 2. Nearest neighbours

Words that appear in similar contexts will have similar vectors.

In [ ]:
for query in ['cat', 'bank', 'river', 'language']:
    print(f"\nNearest neighbours to '{query}':")
    neighbours = nearest_neighbors(model, query, top_k=5)
    for word, sim in neighbours:
        bar = '█' * int(sim * 20)
        print(f"  {word:15s}  {bar:20s}  {sim:.4f}")

## 3. The Famous Analogy: King − Man + Woman ≈ Queen

Our tiny corpus doesn't have "king", "queen" etc, so we'll do this  
with the semantic relations that **do** exist in our corpus —  
and then show the full version with pre-trained vectors.

In [ ]:
# On our tiny corpus: animal-related analogy
# cat is to mat as dog is to ?
print('On our toy corpus:')
print('cat - mat + dog ≈ ?')
results = analogy(model, positive=['cat', 'dog'], negative=['mat'], top_k=3)
for word, score in results:
    print(f'  {word:15s}  similarity={score:.4f}')

In [ ]:
# -------------------------------------------------------------------
# Simulate the famous King - Man + Woman = Queen using pre-trained vectors
# (We'll do it manually with hardcoded example vectors so the notebook
#  works offline without downloading the full GloVe file)
# -------------------------------------------------------------------

# These are APPROXIMATE 3D projections of actual GloVe vectors
# for illustration purposes
royal_demo = {
    'king':   np.array([ 0.50,  0.80,  0.10]),
    'queen':  np.array([ 0.45,  0.75,  0.60]),
    'man':    np.array([ 0.52,  0.20,  0.05]),
    'woman':  np.array([ 0.48,  0.18,  0.55]),
    'prince': np.array([ 0.40,  0.70,  0.08]),
    'princess':np.array([0.37,  0.68,  0.57]),
    'boy':    np.array([ 0.51,  0.12,  0.04]),
    'girl':   np.array([ 0.49,  0.10,  0.53]),
}

# Arithmetic
result_vec = royal_demo['king'] - royal_demo['man'] + royal_demo['woman']

print('Vector arithmetic: King − Man + Woman')
print(f'  Result vector: {result_vec.round(3)}')
print()

# Find closest word
print('Similarity of result vector to all words:')
sims = []
for word, vec in royal_demo.items():
    if word in ('king', 'man', 'woman'):
        continue
    sim = cosine_similarity(result_vec, vec)
    sims.append((word, sim))

sims.sort(key=lambda x: x[1], reverse=True)
for word, sim in sims:
    bar = '█' * int(sim * 30)
    star = ' ← closest match!' if word == sims[0][0] else ''
    print(f'  {word:12s}  {bar:30s}  {sim:.4f}{star}')

In [ ]:
# Visualise the royal analogy in 3D
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

colors = {
    'king': '#185FA5',   'queen': '#185FA5',
    'man':  '#D85A30',   'woman': '#D85A30',
    'prince': '#1D9E75', 'princess': '#1D9E75',
    'boy': '#BA7517',    'girl': '#BA7517',
}

for word, vec in royal_demo.items():
    ax.scatter(*vec, color=colors[word], s=80, alpha=0.8)
    ax.text(vec[0]+0.01, vec[1]+0.01, vec[2]+0.01, word, fontsize=9)

# Draw the analogy arrow
start = royal_demo['king']
end   = result_vec
ax.quiver(*start, *(end - start), color='gray', alpha=0.5, arrow_length_ratio=0.15)
ax.scatter(*result_vec, color='red', s=120, marker='*', zorder=5, label='king−man+woman')

ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.set_zlabel('Dim 3 ("feminine")')
ax.set_title('Word2Vec Analogy Arithmetic (3D demo)', fontsize=11, pad=12)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/02_analogy_3d.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. PCA Projection of Our Trained Vectors

In [ ]:
# All words in our vocabulary
all_words = list(wv.key_to_index.keys())

plot_embeddings(
    model,
    words=all_words,
    title='Word2Vec Embedding Space — PCA Projection (trained on toy corpus)',
    save_path='../outputs/02_word2vec_pca.png',
)

## 5. The Static Embedding Problem — 'bank' has ONE vector

This is the critical limitation that motivates BERT.

In [ ]:
# Both of these sentences are in our corpus
sent_river   = "She sat on the grassy bank watching the river flow slowly."
sent_finance = "The financial bank closed early on the national holiday."

# In Word2Vec, 'bank' has ONE fixed vector regardless of which sentence
bank_vec = wv['bank']

print('With Word2Vec (static embeddings):')
print(f"  'bank' in '{sent_river[:40]}...'")
print(f"  → vector: {bank_vec[:6].round(3)} ...")
print()
print(f"  'bank' in '{sent_finance[:40]}...'")
print(f"  → vector: {bank_vec[:6].round(3)} ... (SAME VECTOR!)")
print()
print('This is the problem BERT was designed to solve.')
print('The next notebook will show that BERT gives "bank" DIFFERENT vectors')
print('depending on which sentence it appears in.')

In [ ]:
# Visualise: 'bank' sits at ONE point, equally distant from 'river' and 'money'
words_to_plot = ['bank', 'river', 'water', 'money', 'loan', 'cat', 'dog', 'language']
words_in_vocab = [w for w in words_to_plot if w in wv.key_to_index]

vecs = np.array([wv[w] for w in words_in_vocab])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vecs)

fig, ax = plt.subplots(figsize=(8, 5))
colors_map = {
    'bank': '#D85A30',
    'river': '#185FA5', 'water': '#185FA5',
    'money': '#1D9E75', 'loan': '#1D9E75',
    'cat': '#888780',   'dog': '#888780',
    'language': '#888780',
}
for i, word in enumerate(words_in_vocab):
    c = colors_map.get(word, '#888780')
    size = 120 if word == 'bank' else 70
    ax.scatter(coords[i, 0], coords[i, 1], color=c, s=size, zorder=3)
    ax.annotate(word, (coords[i, 0] + 0.01, coords[i, 1] + 0.01), fontsize=10)

ax.set_title('Static Embeddings: "bank" sits at ONE point in space', fontsize=11)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.3)

# Annotations
ax.annotate('', xy=(0.05, 0.95), xycoords='axes fraction',
    xytext=(0.05, 0.99), textcoords='axes fraction',
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D85A30', markersize=9, label='ambiguous word'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#185FA5', markersize=9, label='water/geography sense'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1D9E75', markersize=9, label='finance sense'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('../outputs/02_static_embedding_bank.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Summary

| | N-grams | Word2Vec / GloVe |
|---|---|---|
| **Word representation** | Discrete symbols | Dense vectors |
| **Semantic understanding** | ❌ | ✅ partial |
| **Analogy arithmetic** | ❌ | ✅ King − Man + Woman ≈ Queen |
| **Handles polysemy** | ❌ | ❌ (one vector per word) |

**Next:** We'll see how BERT generates a **different vector for the same word** depending on context.